# v2 · 03 · Learning from every month (sliding windows)
v1 learns from **one example per customer**: six months → did they default in October?
But the six months themselves contain more lessons: *after these three months, was the customer late the next month?*

Each customer gives three "practice" examples: months 1–3 → late in month 4, months 2–4 → month 5, months 3–5 → month 6.
A model trained on these (~72,000 examples, no default label used) is then applied to months 4–6 to estimate **"probability of being late next month"**.
We test, with v1's full protocol, whether this new feature improves default prediction.

**Setup.**

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))      # v2 package (adds v1 src to the path too)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from credit_kg_v2 import load_config, OUT
from credit_kg import config as C, scoring as S, modeling as M
import json, pickle
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from credit_kg import data as D
from credit_kg_v2 import sliding as SW

**Load v1's prepared data and fixed split.**

In [2]:
wide, monthly, split = D.load_processed()
w = wide.merge(split)
print(len(w), "customers")

30000 customers


**What one window looks like** (customer 1, months 1–3 → was it late in month 4?).
We use simple per-month flags (late / paid in full / no use) instead of raw status codes, because code `1` is recorded almost only in September and would confuse a model trained on earlier months.

In [3]:
tw_all = SW.training_windows(w)
tw_all[tw_all.customer_id == 1].T

,0,30000,60000
customer_id,1.000000,1.000000,1.000000
window_end,3.000000,4.000000,5.000000
log_limit,9.903488,9.903488,9.903488
late_1,0.000000,0.000000,0.000000
paid_full_1,0.000000,0.000000,1.000000
no_use_1,1.000000,1.000000,0.000000
util_1,0.000000,0.000000,0.000000
pay_rel_1,0.000000,0.000000,0.000000
late_2,0.000000,0.000000,0.000000
paid_full_2,0.000000,1.000000,1.000000


**Practice examples from training customers only**, and how often each 'next month' was late.

In [4]:
tw = SW.training_windows(w[w.split == "train"])
tw.groupby("window_end").agg(examples=("label_late_next", "size"), late_next_month=("label_late_next", "mean")).round(3)

,examples,late_next_month
window_end,,
3,24000,0.140
4,24000,0.148
5,24000,0.229


**Does a model trained on earlier months predict a later month?** (a check that the patterns hold over time)
Train on windows ending in months 3–4, test on the window ending in month 5 (predicting month 6). Compared with the simple rule "late now → late next month".

In [5]:
cols = SW.feature_cols(tw)
early, later = tw[tw.window_end.isin([3, 4])], tw[tw.window_end == 5]
m = SW.make_model().fit(early[cols], early.label_late_next)
auc_model = roc_auc_score(later.label_late_next, m.predict_proba(later[cols])[:, 1])
auc_rule = roc_auc_score(later.label_late_next, later.late_3)
print(f"predicting month-6 lateness: model AUC {auc_model:.3f} vs simple rule {auc_rule:.3f}")

predicting month-6 lateness: model AUC 0.786 vs simple rule 0.780


**Build the new feature for every customer.** Test customers are scored by a model trained on all training customers' windows. Training customers are scored **out-of-fold by customer** (5 groups), so nobody is scored by a model that saw their own windows.

In [6]:
sw = SW.next_late_feature(wide, split)
sw.rename_axis("customer_id").reset_index().to_parquet(OUT / "sw_next_late_prob.parquet", index=False)
sw.describe().round(3)

count    30000.000
mean         0.262
std          0.344
min          0.004
25%          0.045
50%          0.081
75%          0.263
max          0.996
Name: sw_next_late_prob, dtype: float64

**How well does it predict default on its own?** (It was never trained on default.)

In [7]:
table = M.load_table().merge(sw.rename_axis("customer_id").reset_index(), on="customer_id")
for p in ["train", "test"]:
    t = table[table.split == p]; print(f"{p}: AUC of the new feature alone = {roc_auc_score(t[C.TARGET], t.sw_next_late_prob):.4f}")

train: AUC of the new feature alone = 0.7485
test: AUC of the new feature alone = 0.7406


**How new is it?** Its strongest correlation with any existing baseline or graph feature (close to 1 = mostly a repeat).

In [8]:
tr = table[table.split == "train"]
fs = M.feature_sets(); others = sorted(set(fs["B"]))
c = tr[others].corrwith(tr.sw_next_late_prob, method="spearman").abs().sort_values(ascending=False)
c.head(5).round(3).to_frame("abs spearman with new feature")

,abs spearman with new feature
late_recent3,0.757
late_months,0.720
g_max_late_streak,0.719
g_late_onsets,0.634
max_status,0.626


## Does it improve default prediction? (v1 protocol)
Same split, same 25 cross-validation folds, same tree settings, OOF threshold, test set used once, paired bootstrap.
Compared: **A** (v1 baseline) vs **A + new feature**, for the Random Forest and for logistic regression.

In [9]:
params = json.load(open(C.MODELS / "rf_params.json"))
def rf():
    mm = M.make_rf(params["min_samples_leaf"]); mm.set_params(max_features=params["max_features"]); return mm
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
colsA = fs["A"]; colsA_sw = colsA + ["sw_next_late_prob"]
tr, te = table[table.split == "train"], table[table.split == "test"]
y, yt = tr[C.TARGET].to_numpy(), te[C.TARGET].to_numpy()
n_tr, n_te = int(len(y) * .8), int(len(y) * .2)
cv_rf = M.repeated_cv(tr[colsA_sw], y, rf); cv_lr = M.repeated_cv(tr[colsA_sw], y, lr)
cvA_rf, cvA_lr = pd.read_csv(C.TABLES / "cv_folds_A.csv"), pd.read_csv(C.TABLES / "cv_folds_LR-A.csv")
tests = {"RF: A+new - A": M.corrected_resampled_ttest(cv_rf.roc_auc - cvA_rf.roc_auc, n_tr, n_te),
         "LR: A+new - A": M.corrected_resampled_ttest(cv_lr.roc_auc - cvA_lr.roc_auc, n_tr, n_te)}
display(pd.DataFrame({"RF A": [cvA_rf.roc_auc.mean()], "RF A+new": [cv_rf.roc_auc.mean()], "LR A": [cvA_lr.roc_auc.mean()], "LR A+new": [cv_lr.roc_auc.mean()]}, index=["CV ROC-AUC"]).round(4))
pd.DataFrame(tests).T.round(4)

,RF A,RF A+new,LR A,LR A+new
CV ROC-AUC,0.788,0.7883,0.7665,0.767


,mean_diff,folds_b_better,folds,t,p_value
RF: A+new - A,0.0003,14.0,25.0,0.3389,0.7376
LR: A+new - A,0.0006,25.0,25.0,5.7103,0.0000


**Test set, used once:** thresholds chosen on out-of-fold predictions, then the paired bootstrap for the difference.

In [10]:
rows, boots = {}, {}
for name, mk in [("RF", rf), ("LR", lr)]:
    preds = {}
    for label, cols_ in [("A", colsA), ("A+new", colsA_sw)]:
        thr = M.best_f1_threshold(y, M.oof_proba(tr[cols_], y, mk))
        preds[label] = mk().fit(tr[cols_], y).predict_proba(te[cols_])[:, 1]
        rows[f"{name} {label}"] = M.evaluate(yt, preds[label], thr)
    boots[name] = M.paired_bootstrap(yt, preds["A"], preds["A+new"])
    b = boots[name]; print(f"{name}: ΔROC-AUC {b['delta_roc_auc']:+.4f}  95% CI [{b['roc_auc_ci'][0]:+.4f}, {b['roc_auc_ci'][1]:+.4f}]   ΔPR-AUC {b['delta_pr_auc']:+.4f}  CI [{b['pr_auc_ci'][0]:+.4f}, {b['pr_auc_ci'][1]:+.4f}]")
res = pd.DataFrame(rows).loc[["roc_auc", "pr_auc", "precision", "recall", "f1"]]
res.round(4)

RF: ΔROC-AUC -0.0013  95% CI [-0.0037, +0.0011]   ΔPR-AUC -0.0032  CI [-0.0059, -0.0004]


LR: ΔROC-AUC +0.0001  95% CI [-0.0005, +0.0007]   ΔPR-AUC -0.0023  CI [-0.0041, -0.0005]


,RF A,RF A+new,LR A,LR A+new
roc_auc,0.7800,0.7787,0.7573,0.7574
pr_auc,0.5575,0.5543,0.5021,0.4998
precision,0.4975,0.5109,0.4827,0.4906
recall,0.5961,0.5818,0.5576,0.5516
f1,0.5423,0.5440,0.5175,0.5193


**Save** the comparison.

In [11]:
res.to_csv(OUT / "sliding_window_comparison_test.csv")
json.dump({"cv_tests": tests, "bootstrap": boots}, open(OUT / "sliding_window_comparison.json", "w"), indent=2, default=float)
print("saved")

saved


### Takeaways
* **The practice examples work.** A model trained on earlier months predicts later-month lateness (AUC 0.786, slightly better than the simple rule "late now, late next month" at 0.780).
* **Strong on its own:** the new "late next month" probability predicts October default with AUC 0.74–0.75, without ever seeing the default label.
* **No meaningful gain on top of v1's features:**
  - Random Forest: test ΔROC-AUC −0.0013 (95% CI −0.0037 to +0.0011);
  - logistic regression: +0.0006 in cross-validation (better in all 25 folds, but tiny), +0.0001 on test (CI includes 0);
  - PR-AUC even dips slightly for both.
* **Why:** the feature mostly repackages recent lateness (correlation 0.76 with late months in the last quarter). It is informative but redundant, the same pattern as the graph features in v1.
* **Conclusion:** more training examples from the same six months do not add new information. This supports v1's finding that the ~0.78–0.79 ceiling comes from the data.